# Phase 4–10 · End-to-End Generation Pipeline

Executes all generation phases sequentially:

| Phase | What it does |
|---|---|
| **4** | Synthetic account generation — SMOTE learns from `data/original/accounts.csv` |
| **5** | Core transaction generation — SMOTE learns from `data/interim/transactions.parquet` |
| **6** | AML pattern injection — 11 fraud scenarios labelled and appended |
| **7** | Transaction enrichment — account metadata joined onto every row |
| **8** | Feature engineering — temporal, velocity, geographic, transmode features |
| **9** | Constraint validation — schema and business-rule checks |
| **10** | Dataset assembly — final CSVs + JSON logs written to `data/generated/` |

> **Prerequisites**: Run `phase2_cleaning.ipynb` first so that
> `data/original/accounts.csv` and `data/interim/transactions.parquet` exist.
>
> **Scale**: For local testing use the defaults below.  For the full Kaggle run
> set `N_ACCOUNTS = 50_000` and `N_TRANSACTIONS = 5_000_000`.

In [ ]:
# ── Cell 1 · Environment setup ────────────────────────────────────────────────
import os, sys, logging, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings('ignore')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger('generation_pipeline')

# Resolve project root (works locally and on Kaggle)
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    PROJECT_ROOT = Path('/kaggle/working/neural-sentinel')
else:
    PROJECT_ROOT = Path(os.getcwd()).resolve()
    while not (PROJECT_ROOT / 'AGENTS.md').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR   = PROJECT_ROOT / 'data' / 'interim'
GENERATED_DIR = PROJECT_ROOT / 'data' / 'generated'
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Generated dir: {GENERATED_DIR}')
print('\n✓ Environment ready')

In [ ]:
# ── Cell 2 · Configuration ────────────────────────────────────────────────────
# Local / smoke-test defaults.  Scale up for the full Kaggle run.
N_ACCOUNTS     = 10_000
N_TRANSACTIONS = 100_000
SEED           = 42

# AML injections per scenario (11 scenarios total).
# Rule of thumb: ~1 injection per 1 000 transactions, min 100.
N_INJECTIONS = max(100, N_TRANSACTIONS // 1_000)

print(f'Accounts      : {N_ACCOUNTS:,}')
print(f'Transactions  : {N_TRANSACTIONS:,}')
print(f'AML injections: {N_INJECTIONS} per scenario')

In [ ]:
# ── Cell 3 · Load knowledge base (Phase 3.5) ─────────────────────────────────
from src.generation.core.knowledge_extractor import load_knowledge_base

knowledge = load_knowledge_base(PROJECT_ROOT)
print(f'✓ Knowledge base loaded — {len(knowledge)} modules')

In [ ]:
%%time
# ── Cell 4 · Phase 4 — Synthetic Account Generation (SMOTE) ──────────────────
from src.generation.core.account_generator import AccountGenerator

account_gen = AccountGenerator(knowledge, seed=SEED)

# SMOTE fits on data/original/accounts.csv, generates N_ACCOUNTS new rows,
# merges with the original seed, and saves to data/generated/synthetic_accounts.csv.
synthetic_accounts = account_gen.generate(
    n=N_ACCOUNTS,
    merge_with_original=True,     # prepend originals; adds data_source column
    # output_path=None            # auto: data/generated/synthetic_accounts.csv
    # original_data_path=None     # auto: data/original/accounts.csv
)

print(f'\n✓ Accounts ready: {len(synthetic_accounts):,} total rows')
if 'data_source' in synthetic_accounts.columns:
    vc = synthetic_accounts['data_source'].value_counts()
    for label, count in vc.items():
        print(f'   {label:12s}: {count:,}')
if 'is_mule' in synthetic_accounts.columns:
    print(f'   mule rate   : {synthetic_accounts["is_mule"].mean():.2%}')
synthetic_accounts.head(3)

In [ ]:
%%time
# ── Cell 5 · Phase 5 — Core Transaction Generation (SMOTE) ───────────────────
from src.generation.core.transaction_generator import TransactionGenerator

tx_gen = TransactionGenerator(knowledge, synthetic_accounts, seed=SEED)

# SMOTE fits on data/interim/transactions.parquet, generates N_TRANSACTIONS
# new rows, merges with the original seed, and saves to
# data/generated/synthetic_transactions.csv.
core_transactions = tx_gen.generate(
    n=N_TRANSACTIONS,
    merge_with_original=True,     # prepend originals; adds data_source column
    # output_path=None            # auto: data/generated/synthetic_transactions.csv
    # original_data_path=None     # auto: data/interim/transactions.parquet
)

print(f'\n✓ Transactions ready: {len(core_transactions):,} total rows')
if 'data_source' in core_transactions.columns:
    vc = core_transactions['data_source'].value_counts()
    for label, count in vc.items():
        print(f'   {label:12s}: {count:,}')
if 'is_cross_border' in core_transactions.columns:
    print(f'   cross-border: {core_transactions["is_cross_border"].mean():.2%}')
core_transactions.head(3)

In [ ]:
%%time
# ── Cell 6 · Phase 6 — AML Pattern Injection ─────────────────────────────────
from src.generation.core.aml_pattern_injector import AMLPatternInjector

injector = AMLPatternInjector(config={'num_injections': N_INJECTIONS, 'seed': SEED})
fraud_transactions = injector.inject_all_patterns(core_transactions, synthetic_accounts)

print(f'\n✓ AML injection complete')
print(f'   Total rows  : {len(fraud_transactions):,}')
print(f'   Fraud rate  : {fraud_transactions["is_fraud"].mean():.2%}')
display(fraud_transactions['fraud_type'].value_counts(dropna=False))

In [ ]:
%%time
# ── Cell 7 · Phase 7 — Transaction Enrichment ────────────────────────────────
from src.generation.core.enricher import TransactionEnricher

enricher = TransactionEnricher(synthetic_accounts)
enriched_transactions = enricher.enrich(fraud_transactions)

print(f'\n✓ Enrichment complete')
print(f'   Rows   : {len(enriched_transactions):,}')
print(f'   Columns: {len(enriched_transactions.columns)}')
enriched_transactions.head(3)

In [ ]:
%%time
# ── Cell 8 · Phase 8 — Feature Engineering ───────────────────────────────────
from src.generation.core.feature_engineer import FeatureEngineer

engineer = FeatureEngineer()
feature_transactions = engineer.engineer(enriched_transactions)

print(f'\n✓ Feature engineering complete')
print(f'   Rows   : {len(feature_transactions):,}')
print(f'   Columns: {len(feature_transactions.columns)}')
feature_transactions.head(3)

In [ ]:
%%time
# ── Cell 9 · Phase 9 — Constraint Validation ─────────────────────────────────
from src.generation.core.validator import ConstraintValidator

validator = ConstraintValidator(synthetic_accounts, knowledge=knowledge, drop_invalid=False)
val_accounts,     acc_report = validator.validate_accounts()
val_transactions, tx_report  = validator.validate_transactions(feature_transactions)

print('\n✓ Validation complete')
print(f'   Accounts passed    : {acc_report.get("total_rows", 0) - acc_report.get("total_violations", 0):,} / {acc_report.get("total_rows", 0):,}')
print(f'   Transactions passed: {tx_report.get("rows_passed", 0):,} / {tx_report.get("total_rows", 0):,}')
if tx_report.get('rows_failed', 0) > 0:
    print(f'   ⚠ Failed rows: {tx_report["rows_failed"]:,}')

In [ ]:
%%time
# ── Cell 10 · Phase 10 — Dataset Assembly ────────────────────────────────────
from src.generation.core.dataset_builder import DatasetBuilder

builder = DatasetBuilder(GENERATED_DIR)
paths = builder.build(
    accounts=val_accounts,
    transactions=val_transactions,
    schema_report=None,
    validation_report={'accounts': acc_report, 'transactions': tx_report},
)

print('\n✓ Dataset assembled — files written to data/generated/:')
for name, path in paths.items():
    size_mb = path.stat().st_size / 1_048_576
    print(f'   {name:<30s} {path.name}  ({size_mb:.1f} MB)')